# SkillRadar — Del 5: Evaluering, eksperimenter og resultater
## Detaljeret eksamensnotebook (02830 Advanced Project in Digital Media Engineering)

**Formål:** Efter denne del skal du kunne forklare præcis, hvordan I testede SkillRadar, hvad hvert mål betyder, hvad jeres *faktiske* resultater var, og hvilke konklusioner der **ikke** følger af resultaterne.

**Kilder:** SkillRadars `evaluation/`-filer, `skillradar/llm_matcher.py`, `skillradar/variants.py`, `skillradar/study_logging.py`, `README.md` samt `evaluation/results/` på GitHub, kontrolleret 5. november 2026. Link til konkrete kildefiler findes til sidst.

**Notation i denne notebook:**
- **Faktisk projektsresultat** = tal direkte fra committed resultatfiler på GitHub.
- **Faktisk kode** = gengivet fra eller præcist forklaret ud fra projektrepositoriet.
- **Kørbar illustration** = en *selvstændig undervisningsøvelse* på syntetiske data. Disse tal er **ikke** jeres målinger.

De kørbare øvelser bruger standard-Python samt `pandas` og `numpy`. Der kræves ingen Gemini-API-nøgle, personlige CV'er eller Kaggle-datasæt.

## 1. Hvad skulle jeres evaluering bevise?

SkillRadar har tre mål med **forskellige evalueringsspørgsmål**:

| Mål | Hvad testes? | Relevant dokumentation |
|---|---|---|
| **Goal 1: Extraction** | Finder modellen de navngivne teknologier, et menneske finder i jobopslag/CV'er? Gør aliasregler, LLM eller hybrid en forskel? | `evaluation/evaluate.py`, labels, `evaluation/results/goal1_test_*.md` |
| **Goal 2: Robust demand** | Er efterspørgselsrangeringerne nogenlunde ens i to uafhængige datasæt? Er de øverste pladser stabile? Skiller junior-/senioropslag sig ud? | `analysis/goal2.py`, `skillradar/robustness.py`, `analysis/results/` |
| **Goal 3: Prototype** | Kan studerende forstå, korrigere og anvende en skill-gap-rapport? | `evaluation/user_study.md`, `app_testing.py`, `skillradar/study_logging.py` |

**Hvorfor opdele det?** En god F1-score betyder ikke nødvendigvis, at anbefalingerne er nyttige. Og en høj korrelation mellem datasæt garanterer ikke, at de enkelte CV-match er korrekte. Evalueringen må derfor måle flere led.

Denne notebook går dybest i Goal 1 og brugertesten. Goal 2-statistikken blev gennemgået i Del 4, men dens faktiske konklusioner opsummeres her, fordi de er del af hele projektets evidens.

## 2. Datagrundlag for Goal 1: 40 jobopslag

**Faktisk kode: `evaluation/sample_postings.py`.** I trækker en reproducerbar stikprøve på **40 data-scientist-opslag** fra den behandlede primary-tabel med `seed=42`:

- **10 dev-postings**: bruges til debugging og til at vælge indstillinger.
- **30 test-postings**: holdes tilbage til den endelige, frosne evaluering.
- `evaluation/manifest.csv` gemmer `item_id`, `split`, `posting_id` og `title`.
- De tilhørende opslagstekster gemmes lokalt under `evaluation/items/` og er ikke offentligt redistribueret.
- Scriptet stopper normalt, hvis manifest allerede eksisterer, så udtrækket ikke ændres ved et uheld.

**Hvorfor seed?** Det giver en gentagelig tilfældig stikprøve. **Hvorfor dev/test?** Fordi tuning på de samme opslag, man rapporterer resultater for, overvurderer kvaliteten (*data leakage*).

**Begrænsning:** Testen på jobopslag er forholdsvis lille og er kun fra én rolle (`data scientist`). Det er ikke automatisk evidens for tilsvarende performance på alle fem roller, CV'er eller andre sprog.

In [ ]:
from IPython.display import display
import pandas as pd
import numpy as np
import math
from collections import Counter

# KØRBAR ILLUSTRATION: opdeling med fast seed; ikke projektets testopslag.
syntetiske_ids = [f"post_{i:03d}" for i in range(1, 41)]
manifest_demo = pd.DataFrame({"item_id": syntetiske_ids,
                              "split": ["dev"]*10+["test"]*30})
print(manifest_demo["split"].value_counts())
print("Overlap mellem dev og test:", set(manifest_demo.loc[manifest_demo.split=='dev','item_id']) & set(manifest_demo.loc[manifest_demo.split=='test','item_id']))

### Forklaring til output

Der er 10 udviklingsitems og 30 testitems uden overlap. Her er det kun illustrationen af opdelingen, ikke selve den tilfældige sampling fra jobdata. I jeres projekt anvendes `DataFrame.sample(n, random_state=seed)` på poolen af relevante jobopslag, før `split` tildeles.

**Alternativer:** k-fold cross-validation, flere forskellige random seeds eller et større, stratificeret testset på tværs af roller. Fordelen er mere robuste konklusioner; ulempen er flere timer med manuel labeling og eventuelt flere API-kald.

## 3. Menneskelige labels: Hvordan lavede I “ground truth”?

Fra `evaluation/GUIDELINES.md`:

1. To personer læser de **samme 40 opslag uafhængigt**, før de ser matcherens output.
2. De noterer alle **navngivne værktøjer og teknologier**: Python, SQL, Tableau, Git osv.
3. Almindelige kompetencer/metoder som “statistics” og “machine learning” tæller ikke som et navngivet tool i denne definition.
4. Virksomhedsnavne uden tool-betydning og almindelige ord (fx “Go further”) tæller ikke.
5. Også værktøjer uden for O*NET-ordforrådet skal skrives ned, fx Jupyter eller `uv`.
6. Hvert værktøj tælles én gang pr. opslag, og usikre tilfælde kan kommenteres.

**Labels-format:** `item_id`, `split`, `labeller`, `true_tools` (semikolonadskilt) og `notes`. For eksempel: `Python; SQL; AWS; Power BI`.

**Hvorfor blind labeling?** Hvis man ser modellens svar først, kan labels utilsigtet blive påvirket af de ord, modellen finder. Den type bias kan give en kunstigt høj evalueringsscore.

In [ ]:
# KØRBAR ILLUSTRATION: to menneskelige lister over et enkelt fiktivt opslag
label_A = {"Python", "SQL", "Power BI"}
label_B = {"Python", "SQL", "Jupyter"}
print("Enighed:", sorted(label_A & label_B))
print("Kun A:", sorted(label_A - label_B))
print("Kun B:", sorted(label_B - label_A))
print("Union til gold:", sorted(label_A | label_B))

### Hvorfor en union til gold?

I `evaluation/merge_gold.py` valgte I **unionen** af de to labelleres fund: hvis mindst én havde fundet en teknologi, kom den med i de samlede labels (efter navnenormalisering). Uenigheder bevares i `notes` og kan efterkontrolleres.

**Begrundelse i kodekommentarerne:** Uenighederne bestod i høj grad af, at den ene labeller havde overset et værktøj, snarere end at de var fundamentalt uenige om en omtale.

**Konsekvens:** Union kan øge antallet af sande værktøjer og sænke recall for en matcher i forhold til et snævrere konsensus-gold. Den kan også inkludere fejl fra én labeller. Et alternativ er adjudikation med eksplicit fælles gennemgang af hvert disagreement, eller kun intersection (sidstnævnte risikerer at udelukke ægte, men svære, værktøjer).

## 4. Hvor enige var I som labellere?

**Faktisk resultat fra `evaluation/GUIDELINES.md`:**

- **40** opslag blev markeret af begge labellere.
- *Positive agreement (F1 mellem labellere)*: **0,86**.
- *Cohen's kappa* over beslutninger om `(opslag, teknologi)`: **0,90**.

Det er to **forskellige** mål. Positive-agreement F1 er særligt informativt om, hvor enige I er om hvilke *positive* teknologier, der findes. Kappa inkluderer også enighed om de mange værktøjer, som **ikke** er i teksten. Jeres egne retningslinjer advarer derfor mod at præsentere kappa alene.

**Konklusion:** I havde betydelig enighed, men labels er ikke fuldstændig objektive. Menneskelige uenigheder sætter en relevant ramme for modellens sammenligning.

In [ ]:
def f1_sets(a,b):
    overlap = len(a & b)
    p = overlap/len(a) if a else 0
    r = overlap/len(b) if b else 0
    return 2*p*r/(p+r) if p+r else 0

print("F1 for det fiktive enkeltopslag:", round(f1_sets(label_A,label_B),3))

**Forklaring:** F1 her er en *illustration* for én syntetisk tekst. Den skal ikke forveksles med projektets 0,86, der er beregnet over 40 rigtige opslag. Mulige alternativer er Jaccard-similaritet og eksplicit gennemgang af uenigheder ved en tredje annotator.

## 5. Hvad betyder TP, FP og FN?

Modellens fund sammenlignes med de menneskelige labels:

- **True positive (TP):** matcher finder en teknologi, der står i gold.
- **False positive (FP):** matcher finder en teknologi, som ikke står i gold.
- **False negative (FN):** teknologien står i gold, men matcher overser den.

Eksempel: Hvis gold = `{Python, SQL, Docker}` og modellen finder `{Python, SQL, Java}`, så er `TP=2`, `FP=1` og `FN=1`.

I projektet arbejdes der med `(item, tool)`-par, og jeres hovedtal er **micro-averaged** over alle relevante par. Det betyder, at hvert fund/forsømt værktøj tæller, snarere end at hvert opslag først får samme vægt i en gennemsnitlig F1.

In [ ]:
truth = {"Python","SQL","Docker"}
found = {"Python","SQL","Java"}
tp, fp, fn = len(found & truth), len(found-truth), len(truth-found)
print({"TP":tp,"FP":fp,"FN":fn})
precision = tp/(tp+fp)
recall = tp/(tp+fn)
f1 = 2*precision*recall/(precision+recall)
print(f"Precision={precision:.3f}; Recall={recall:.3f}; F1={f1:.3f}")

## 6. Precision, recall og F1 – helt præcist

\[
\mathrm{Precision}=\frac{TP}{TP+FP},\quad
\mathrm{Recall}=\frac{TP}{TP+FN},\quad
F_1=2\frac{\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}.
\]

**Precision:** Af alle værktøjer vi *påstår* at have fundet, hvor mange er korrekte? Høj precision betyder få falske fund.

**Recall:** Af alle værktøjer der *faktisk* var i teksten, hvor mange fandt vi? Høj recall betyder få oversete værktøjer.

**F1:** Harmonisk gennemsnit, der straffer en metode, hvis en af de to målinger er lav.

**Hvorfor begge?** En metode kan finde næsten alle værktøjer ved også at gætte på en masse forkerte. Det giver høj recall men dårlig precision. Omvendt kan man være konservativ og misse mange værktøjer.

**Konsekvens for SkillRadar:** FP kan give en bruger indtryk af at have en teknologi på CV'et, som ikke er der. FN kan få en eksisterende teknologi til at fremstå som en “manglende skill”.

## 7. Vocab recall versus all recall: jeres centrale forskel

I bruger O*NET som et **lukket ordforråd** (vocabulary) i keyword-matcheren. Men de menneskelige labellere markerer også værktøjer uden for dette katalog (*out of vocabulary*, OOV).

`evaluation/evaluate.py` beregner to recall-mål:

- **Recall (vocab):** andelen af korrekt fundne gold-værktøjer, som **findes i O*NET-vokabularet**.
- **Recall (all):** andelen af korrekt fundne gold-værktøjer, inklusive dem **uden for** O*NET.

**Faktisk testresultat:** I de 30 held-out opslag er der **204 labellede værktøjsforekomster**, hvoraf **47** (23 %) er OOV. Den frosne keyword-matcher kan ikke udtrække disse 47 som OOV, så dens `recall (all)` bliver lavere end `recall (vocab)`.

Det er ikke blot et problem med tokenisering; det er en **begrænsning ved, hvilke værktøjer modellen overhovedet kender**.

In [ ]:
# KØRBAR ILLUSTRATION med faktiske TÆLLINGER fra goal1_test_strict.md, full matcher
TP, FP, FN_vocab, FN_OOV = 148, 4, 9, 47
p = TP/(TP+FP)
r_vocab = TP/(TP+FN_vocab)
r_all = TP/(TP+FN_vocab+FN_OOV)
def harmonic(a,b): return 2*a*b/(a+b) if a+b else 0
print(f"Precision:      {p:.3f}")
print(f"Recall vocab:   {r_vocab:.3f}")
print(f"Recall all:     {r_all:.3f}")
print(f"F1 vocab:       {harmonic(p,r_vocab):.3f}")
print(f"F1 all:         {harmonic(p,r_all):.3f}")

**Hvad fortæller output?** Tallene skal stemme med den commit'ede tabel efter afrunding: precision ca. 0,97, vocab recall ca. 0,94, all recall ca. 0,73. Det er et af de vigtigste forhold at kunne forklare til eksamen.

## 8. Hvad er “strict” og “lenient” scoring?

I `evaluation/evaluate.py` findes to evalueringsdefinitioner:

- **Strict (default):** Et rigtigt match kræver at det fundne værktøj matches som samme kanoniske O*NET-værktøj som gold. OOV-gold tælles som overset for strict scoring.
- **Lenient (`--lenient`):** Også værktøjer uden for O*NET kan give credit, hvis modellens output-navn kan matches med et OOV-navn efter normalisering af stavning/tegnsætning/vendor-præfikser, fx “Amazon SageMaker” og “SageMaker”. Reglen bruges ens for de sammenlignede metoder.

**Denne forskel er afgørende:** En open-vocabulary LLM kan udtrække navngivne tools, der ikke findes i O*NET. Strict scoring belønner ikke disse fund på samme måde, mens lenient scoring kan gøre det.

Ingen af de to scoringstyper er universelt “den rigtige”. De tester to lidt forskellige mål: ren O*NET-mapping versus bredere udtræk af teknologier.

## 9. De faktiske testresultater — strict scoring (30 opslag)

Fra `evaluation/results/goal1_test_strict.md`:

| Metode | Precision | Recall (vocab) | Recall (all) | F1 (all) | TP / FP |
|---|---:|---:|---:|---:|---:|
| **Full keyword** | **0,97** | 0,94 | 0,73 | **0,83** | 148 / 4 |
| LLMOnly | 0,78 | 0,95 | 0,73 | 0,75 | 149 / 42 |
| Hybrid | 0,77 | 0,97 | 0,75 | 0,76 | 153 / 46 |
| HybridVerify | 0,98 | 0,94 | 0,73 | 0,83 | 148 / 3 |

**Forklaring:** I strict scoring fremstår LLMOnly og Hybrid langt dårligere i precision end keyword, primært fordi mange af deres OOV-outputs **ikke tæller som korrekte** efter denne scoringsdefinition. Dette er ikke det samme som at bevise, at LLM'erne rent faktisk hallucinerede alle disse værktøjer.

**Eksamensfælden:** Sig ikke bare “Hybrid havde precision 0,97”. Det tal tilhører **lenient** sammenligningen. Under **strict** er tallet **0,77**.

## 10. De faktiske testresultater — lenient scoring (30 opslag)

Fra `evaluation/results/goal1_test_lenient.md`:

| Metode | Precision | Recall (vocab) | Recall (all) | F1 (all) | TP / FP |
|---|---:|---:|---:|---:|---:|
| Full keyword | 0,97 | 0,94 | 0,73 | 0,83 | 148 / 4 |
| LLMOnly | 0,98 | 0,96 | 0,92 | 0,95 | 188 / 3 |
| **Hybrid** | **0,97** | **0,97** | **0,95** | **0,96** | 193 / 6 |
| HybridVerify | 0,98 | 0,94 | 0,73 | 0,83 | 148 / 3 |

**Konklusion under lenient scoring:** Hybrid genfinder flere af de 204 labellede værktøjer og får en `recall (all)` på 0,95. Den har samme afrundede precision som keyword (0,97). Forbedringen i all recall er **+0,22** (22 procentpoint) med rapporteret 95 % paired-bootstrap-interval **+0,15 til +0,29**.

**Vigtigt metodisk skel:** Det er en forbedring på *held-out jobopslag*, evalueret efter den valgte lenient matchingregel. Det er ikke dokumentation for tilsvarende forbedring på personlige CV'er, eller en samlet forbedring af den endelige app, som bruger keyword-matcheren.

In [ ]:
# KØRBART: plot de dokumenterede, afrundede resultater fra resultattabellerne
import matplotlib.pyplot as plt
method_names=["Keyword", "LLMOnly", "Hybrid", "HybridVerify"]
strict_precision=[0.97,0.78,0.77,0.98]
lenient_precision=[0.97,0.98,0.97,0.98]
x=np.arange(len(method_names))
fig, ax=plt.subplots(figsize=(9,4))
ax.bar(x-0.18,strict_precision,0.36,label="Strict")
ax.bar(x+0.18,lenient_precision,0.36,label="Lenient")
ax.set_xticks(x,method_names)
ax.set_ylim(0,1.1)
ax.set_ylabel("Precision")
ax.set_title("Faktiske testtal: scoring ændrer fortolkningen")
ax.legend()
plt.show()

### Hvordan læser man figuren?

De samme modeloutputs bliver vurderet under forskellige regler. En evaluator, der kun ser ét af de to sæt resultater, kan drage en helt anden konklusion.

**Alternative evalueringsmetoder:** Human adjudication af de konkrete OOV-fund, semantisk normalisering med manuelt kontrollerede ækvivalensklasser eller både strict og lenient rapporteret som et interval over rimelige scoringantagelser.

## 11. Ablation studies: Hvilke matcherregler var nyttige?

I `skillradar/variants.py` bruger I fire **kumulative** varianter:

1. `exact`: kun de fulde O*NET-navne.
2. `vendor`: også kortere navne uden producentpræfiks og “software”.
3. `acronym`: også forkortelser såsom SQL og AWS.
4. `full`: også case-regler for korte eller tvetydige navne og øvrig fuld matcherlogik.

**Ablation** betyder, at man fjerner eller tilføjer bestemte komponenter kontrolleret for at måle deres bidrag. Fordi testpostings og gold er de samme, kan I bedre isolere værdien af hver normaliseringsregel.

I sammenligner ikke en vilkårlig ny model mod en anden på forskellige data; I sammenligner **samme grundlæggende matcher** med tiltagende funktionalitet.

## 12. Faktiske ablation-resultater (strict test)

Fra `evaluation/results/goal1_test_ablation.md`:

| Variant | Precision | Recall (vocab) | Recall (all) | F1 (vocab) |
|---|---:|---:|---:|---:|
| Exact | 0,96 | 0,56 | 0,43 | 0,71 |
| Vendor | 0,94 | 0,76 | 0,59 | 0,85 |
| Acronym | 0,95 | **0,95** | 0,73 | 0,95 |
| Full | **0,97** | 0,94 | 0,73 | **0,96** |

**Hovedfund:** Store forbedringer i recall opstår, når ordforrådet kan genkende forkortelser. Sammenlignet med exact stiger vocab recall fra 0,56 til 0,95 i acronym-varianten, altså **+0,39** (39 procentpoint).

**Nuance:** `full` har en lille smule lavere vocab recall end `acronym` (0,94 mod 0,95), men bedre precision (0,97 mod 0,95). Den fulde variant er derfor ikke “bedst på alle mål”.

In [ ]:
ablation = pd.DataFrame({
    "variant":["exact","vendor","acronym","full"],
    "precision":[0.96,0.94,0.95,0.97],
    "recall_vocab":[0.56,0.76,0.95,0.94],
    "recall_all":[0.43,0.59,0.73,0.73]
})
display(ablation)
print("Effekt af acronym vs exact (vocab recall, procentpoint):",
      round((ablation.loc[2,"recall_vocab"]-ablation.loc[0,"recall_vocab"])*100))

**Outputtet:** 39 procentpoint ekstra vocab recall; det er **ikke** en forbedring på 39 %. Forskellen mellem “procentpoint” og “relativ procentvis stigning” er et oplagt eksamensspørgsmål.

## 13. LLMOnly, Hybrid og HybridVerify: Hvad evaluerede I?

**Faktisk kode i `skillradar/llm_matcher.py`:**

- `LLMOnly`: Gemini læser tekst og udtrækker værktøjer med et kort citeret tekststykke for hvert værktøj.
- `Hybrid`: keyword-matcheren foreslår kandidater; Gemini **bekræfter/afviser** kandidater i kontekst og **tilføjer** oversete værktøjer.
- `HybridVerify`: bruger samme verifikationssvar som Hybrid, men **ignorerer alle tilføjelser**. Den isolerer, om kontekstkontrollen alene gør en forskel.

**Fælles struktur:** prompting med labellingregler, JSON-schema, grounded citatkontrol og linking tilbage til O*NET-navne eller et `OOV:`-navn. LLM'en trænes ikke på jeres data i denne kode; I anvender en eksisterende model med prompt-/few-shot-konfiguration.

**Frossen evalueringskonfiguration:** `gemini-3.5-flash-lite`, few-shot fra `post_003` og `post_005`, embedding-link-threshold `1.0` (embedding linking slået fra). Disse værdier står i `evaluation/llm_frozen_settings.txt`. Default-variabler i den aktuelle kode kan afvige, så I bør altid skelne mellem **evalueret opsætning** og **nuværende default**.

## 14. Hvad viser HybridVerify-eksperimentet?

I den lenient test:
- Full keyword: precision 0,97 og recall (all) 0,73.
- HybridVerify: precision 0,98 og recall (all) 0,73.
- Hybrid: precision 0,97 og recall (all) 0,95.

I `goal1_test_lenient.md` står, at Hybrid kun havde **1 afvist kandidat** i de 30 testopslag, og HybridVerify var bedre på ét opslag og uændret på 29 (målt på antal fejl pr. item).

**Tolkning:** I denne test kommer næsten hele hybridens forbedrede dækning fra dens evne til at *tilføje manglende værktøjer*, ikke fra at afvise en masse forkerte keyword-kandidater.

**Begrænsning:** Én afvisning i 30 opslag fortæller ikke nødvendigvis, hvordan modellen vil opføre sig på helt anderledes CV-tekst eller domæner.

In [ ]:
# KØRBAR ILLUSTRATION af ablation af LLM-tilføjelser (IKKE rigtige modeloutputs)
candidates={"Python","SQL"}
verified={"Python","SQL"}            # LLM beholdt begge
additional={"Databricks","Jupyter"}  # LLM foreslog ekstra i eksemplet
hybrid_verify=verified
hybrid=verified|additional
print("HybridVerify:",sorted(hybrid_verify))
print("Hybrid:",sorted(hybrid))
print("Tilføjelsernes bidrag:",sorted(hybrid-hybrid_verify))

**Forklaring:** Ved at holde `verified` konstant og tænde/slukke for `additional` isolerer vi værdien af den del, der finder ekstra værktøjer. `HybridVerify` er derfor en metodisk vigtig kontrol, ikke en selvstændig ny model.

## 15. Grounding: Hvordan kontrollerer I LLM-svar?

For hvert foreslået værktøj kræver LLM-pipelinen en **quote** fra teksten. `grounded(quote, text)` kontrollerer, at citatet findes i teksten efter mindre normalisering. Hvis citatet ikke findes, kan værktøjet stadig bevares, hvis **navnet selv** kan findes som selvstændige ord i teksten (`find_quote`); ellers droppes det.

Dette er en **hallucination guard**, men ikke en fuld garanti for korrekt fortolkning. Et ord kan stå i teksten uden at have betydningen af et anvendt værktøj; og et værktøj kan være nævnt uden at være et kompetencekrav.

**Faktisk testresultat:** I den lenient-kørte Hybrid-tabel er `dropped_ungrounded = 1`. Den tabel viser også `linked_embedding = 0`, i overensstemmelse med den frosne threshold på 1.0.

In [ ]:
# Enkel undervisningsillustration af en tekstbaseret grounding-check
text="We use Python and Docker for production." 
def normal(s): return " ".join(s.lower().split())
for quote in ["Python and Docker", "TensorFlow", "We use"]:
    print(f"{quote!r} findes i teksten?", normal(quote) in normal(text))

**Nuance:** Ovenstående er en forenklet demonstration og **ikke** nøjagtigt den samme `grounded()`-funktion. Den faktiske funktion håndterer typografiske apostroffer/citater, mellemrum og ellipser. Det illustrerer alene kontrollens idé.

## 16. Hvorfor slog I embedding-linking fra?

Fra `evaluation/llm_frozen_settings.txt`:

- Link-threshold blev valgt på **dev**, ikke på test.
- Den frosne opsætning bruger `SKILLRADAR_LINK_THRESHOLD=1.0`, så embedding-mapping ikke aktiveres.
- I noterer, at de embedding-links, der blev accepteret på dev, reducerede kvaliteten: et nært beslægtet navn er ikke altid **samme teknologi**.

Eksempel på problemet: At to teknologier er semantisk beslægtede betyder ikke, at én kan erstatte den anden. Hvis en model linker `BigQuery` til `MySQL`, kan det fordreje både score og anbefalinger.

**Alternativ:** Bedre embeddings, mere restriktiv threshold, entity-linking med kandidatlister eller menneskelig kontrol af usikre mappings. Den enkle beslutning om at deaktivere en skadelig komponent på dev er metodisk forsvarlig.

## 17. Bootstrap og usikkerhed i Goal 1

I `evaluation/evaluate.py` bruges **2.000 bootstrap-resamples over items** i evalueringsstatistikken. Det betyder, at I gentagne gange trækker opslag med tilbagelægning og genberegner metrics.

**Hvorfor resample opslag frem for enkelte `(opslag, teknologi)`-par?** Teknologier inden for ét opslag er afhængige; hvis et opslag er svært at udtrække fra, kan flere tool-beslutninger fejle samtidigt. Resampling af hele opslag bevarer denne interne afhængighed bedre.

**95 %-intervaller:** 2,5- og 97,5-percentilen af bootstrap-fordelingen bruges som usikkerhedsinterval for målet.

**Paired bootstrap:** Når to modeller testes på samme opslag, trækkes de **samme** opslag i hvert resample. Så kan man beregne forskellen i metrics inden for hvert resample. Det er mere direkte end at sammenligne, om modellernes to separate intervaller overlapper.

In [ ]:
# KØRBAR bootstrap-illustration (syntetiske item-tællinger)
items=[(4,1,0),(2,0,2),(3,1,1),(1,0,2),(5,0,0)] # (TP,FP,FN) pr. item
rng=np.random.default_rng(42)
def micro_f1(selected):
    counts=np.array(selected).sum(axis=0)
    tp,fp,fn=map(int,counts)
    return 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0
scores=[micro_f1([items[j] for j in rng.integers(0,len(items),len(items))]) for _ in range(2000)]
print("Syntetisk F1:",round(micro_f1(items),3))
print("Syntetisk 95 % bootstrap:",np.quantile(scores,[0.025,0.975]).round(3))

**Output:** Et interval på baggrund af fem fiktive items. De faktiske rapporter har 30 postings for Goal 1. Metoden viser variationen fra hvilke items, der bliver udtrukket, men beskytter ikke mod systematiske skævheder i, at de 30 allerede kommer fra én rolle og ét datasæt.

## 18. Hvordan tester I, om Hybrid er bedre end keyword?

Fra den **faktiske paired comparison** under lenient scoring (`goal1_test_lenient.md`), Hybrid minus Full:

- **ΔRecall (all): +0,22**; 95 % CI **+0,15 til +0,29**.
- **ΔPrecision:** afrundet **−0,00**; 95 % CI **−0,02 til +0,02**.
- **ΔF1 (vocab): +0,01**; 95 % CI **−0,00 til +0,03**, rapporteret `p = 0.099`.

**Vigtig præcision:** Den tilhørende paired F1-kolonne er projektets **vocabulary F1**, ikke `F1 (all)`. Derfor kan man **ikke** bruge `p = 0.099` som p-værdi for forbedringen i `F1 (all)`.

I kan sige, at forskellen i `recall (all)` har et rapporteret interval, der ikke indeholder 0, men skal stadig huske på den lille, rolleafgrænsede stikprøve og lenient-definitionen.

## 19. Faktisk CV-evaluering: Kun fire test-CV'er i resultatfilen

Det er vigtigt **ikke** at gentage en tidligere projektstatus, hvis der siden er kommet konkrete resultater. `evaluation/results/goal1_cv_keyword.md` indeholder en test på **4 CV'er**, med 61 labellede værktøjer og 14 OOV (23 %).

| Keyword-variant | Precision | Recall (vocab) | Recall (all) | F1 (all) |
|---|---:|---:|---:|---:|
| Exact | 0,95 | 0,89 | 0,69 | 0,80 |
| Vendor | 0,96 | 0,96 | 0,74 | 0,83 |
| Acronym | 0,96 | 1,00 | 0,77 | 0,85 |
| Full | 0,96 | **1,00** | **0,77** | **0,85** |

**Hvad viser det?** På netop disse fire CV'er fandt Full alle gold-værktøjer, der var i vokabularet, men kunne stadig ikke finde OOV-værktøjerne. Precision var 0,96.

**Hvad viser det ikke?** At recall vil være 100 % for vilkårlige CV'er; fire CV'er er en meget lille stikprøve. Denne fil sammenligner ikke LLM og Hybrid på CV'er, så den kan ikke bruges som dokumentation for deres CV-performance.

## 20. Hastighed, API-kald og hvorfor webappen ikke bruger Hybrid

De lagrede LLM-statistikker fra `goal1_test_lenient.md` viser:

| Metode | Oprindelig samlet svartid på 30 opslag | Sekunder pr. opslag | Input / output tokens |
|---|---:|---:|---:|
| LLMOnly | 649,2 s | **21,64 s** | 86.242 / 5.306 |
| Hybrid | 530,4 s | **17,68 s** | 101.549 / 7.204 |
| HybridVerify | 530,4 s | **17,68 s** | 101.549 / 7.204 |

**Kritisk nuance:** Tabellen viser `api_calls = 0` og `cached = 30`, fordi evalueringen blev genkørt på allerede cachede svar. `seconds`-feltet beskriver de oprindelige API-kald, som cachen gemte; det er **ikke** ventetiden for den cachede genkørsel. `HybridVerify` genbruger samme LLM-svar som Hybrid.

**Designbeslutning:** I webappen er keyword-matching langt hurtigere og kræver ingen ekstern API-nøgle til CV-tekst. Appen lader derefter brugeren kontrollere matchene (human-in-the-loop). Det er et praktisk kompromis mellem kvalitet, privatliv, forklarbarhed og svartid.

## 21. Goal 2: Hvad blev faktisk evalueret om markedet?

Dette er statistik fra `analysis/results/goal2_output.txt`, behandlet detaljeret i **Del 4**.

**Replikation:** Spearman-rankkorrelation mellem to uafhængige LinkedIn-snapshots ved sammenligning af de øverste teknologier:

| Rolle | Spearman ρ |
|---|---:|
| Data analyst | **0,87** |
| Software engineer | **0,87** |
| Data scientist | **0,80** |
| ML engineer | 0,82 (lille sample) |
| UX designer | **0,63** |

**Resultat:** Toprangeringen er relativt sammenlignelig mellem kilderne for de første tre roller, men mindre robust for UX. Dette er evidens for **rangstabilitet på tværs af snapshots**, ikke for en tidsserie eller for udviklingen i efterspørgslen siden 2024.

**Seniority-eksempel:** SQL nævnes i 36,2 % af junior- versus 66,2 % af senior-dataanalytikeropslag i den sekundære kilde (forskel +30,0 procentpoint); forskellen er signifikant efter korrektion for multiple tests. Ikke alle synlige forskelle er signifikante.

## 22. Goal 3: Brugertestens konkrete design

Fra `evaluation/user_study.md`:

- **Pilot:** 2 medstuderende, ca. 15 minutter hver.
- **Peer trial:** planlagt 6–8 jobsøgende studerende efter piloten.
- **Metode:** think-aloud – deltagere siger højt, hvad de tænker, mens de bruger appen.
- **Opgaver:** upload CV og vælg rolle; ret de fundne tools; byg rapport; udpeg 1–2 teknologier at prioritere og forklar hvorfor.
- **Observationer:** gennemførte opgaver med/uden hjælp, forvirring, antal tilføjede/fjernede tools, skriftlig feedback.

Der anvendes seks Likert-spørgsmål på skalaen 1–5 om ekstraktionsnøjagtighed, relevans, nytte, forklarbarhed, tillid og brugervenlighed. To fritekstspørgsmål spørger, hvad der var forvirrende, og hvad der burde forbedres.

**Hvorfor ikke kun en spørgeskema-score?** Think-aloud og opgaveobservationer kan afsløre konkrete misforståelser, selv hvis en deltager giver en høj samlet bedømmelse.

## 23. Hvad ændrede piloten?

Projektets `README.md` beskriver feedback fra de første to deltagere:

- 95 %-intervallerne var uklare.
- Det burde fremgå først, hvilke tools brugerens CV **allerede** indeholder.
- Designet virkede for generisk.

I reagerede ved at ændre rapporten: eksisterende tools vises først, konfidensintervaller forklares med almindeligt sprog, og radargrafen bruges til at vise demand.

**God eksamensforklaring:** Brugercentreret design betyder ikke bare, at man har spurgt nogen. I kan pege på **feedback → konkret ændring i designet**.

**Hvad kilderne ikke dokumenterer her:** En fuldstændig endelig peer-trial-sammenfatning med alle deltagere og alle ratings. README'en, som er læst her, omtaler peer trial som kommende. Hvis I siden har indsamlet flere svar i et privat Google Sheet, er de **ikke** tilgængelige i de offentlige resultatfiler, så der er ikke indsat opdigtede brugerresultater i denne notebook.

## 24. Hvordan håndterer testappen privatliv og data?

`app_testing.py` tilføjer en femte **Feedback**-side til appens fire trin.

`skillradar/study_logging.py` bruger en whitelist over felter såsom `participant_id`, `role`, `dataset`, `detected`, `added`, `removed` og de seks ratings. Der er **ikke** felter til rå CV-tekst eller filnavn. Uventede felter afvises af `_row()`.

Efter samtykke kan resultater gemmes i et Google Sheet (hosted app via Streamlit Secrets) eller i `data/user_study/responses.csv` lokalt. Projektets dokumentation siger, at CV-tekst kun behandles i hukommelsen i webappen.

**Begrænsning:** Dette beskriver, hvad jeres kode er designet til at gøre; det er ikke en uafhængig sikkerheds-/GDPR-audit. En deltager kan også skrive personlige oplysninger i et fritekstfelt, selv om feltet begrænses i længde.

In [ ]:
# KØRBAR ILLUSTRATION af whitelisting (skriver IKKE til disk)
allowed={"participant_id","role","accuracy","trust","added","removed"}
def validate_study_row(row):
    unexpected=set(row)-allowed
    if unexpected:
        raise ValueError(f"Uventede felter: {sorted(unexpected)}")
    return {k:row.get(k,"") for k in allowed}
print(validate_study_row({"participant_id":"P01","role":"data scientist","trust":4}))
try:
    validate_study_row({"participant_id":"P01","cv_text":"Private CV text"})
except ValueError as error:
    print("Afvist:",error)

**Forklaring:** Idéen i den rigtige `_row()` er, at programmet accepterer en bestemt liste over studie-felter og afviser andre. Ovenstående er en mindre undervisningsversion; den rigtige kode tilføjer også UTC-timestamp, normaliserer/trunkerer fritekst og kan skrive til CSV eller Google Sheet.

## 25. Sådan kan I analysere peer trial, hvis I har de faktiske svar

Når I har adgang til responses fra testappen, kan I eksempelvis beskrive:

- Antal gyldige deltagere og antal prøvebesvarelser (`test_run`).
- Antal deltagere der gennemførte hver opgave uden hjælp, baseret på facilitatorens *separate observationsnoter*.
- Median og spredning i Likert-besvarelser (1–5); median er ofte mere informativ end et rent gennemsnit for små, ordinalt skalerede besvarelser.
- Hvor mange teknologier der blev tilføjet/fjernet, og mønstre i begrundelser.
- Tilbagevendende temaer i fritekst-feedback.

**Vigtigt:** `detected`, `added` og `removed` kan være indikatorer på matcherens brugbare nøjagtighed i praksis, men de er **ikke** automatisk det samme som et fuldt manuelt annoteret gold-standard præcision-/recall-resultat. Deltagere kan selv overse fejl.

In [ ]:
# SYNTETISKE STUDIESVAR – IKKE JERES PEER TRIAL
toy_study=pd.DataFrame([
    {"id":"P01","test_run":False,"trust":4,"usefulness":5,"added":1,"removed":0},
    {"id":"P02","test_run":False,"trust":3,"usefulness":4,"added":2,"removed":1},
    {"id":"TEST01","test_run":True,"trust":5,"usefulness":5,"added":0,"removed":0},
    {"id":"P03","test_run":False,"trust":4,"usefulness":3,"added":0,"removed":0},
])
real_rows=toy_study.loc[~toy_study.test_run]
print("Syntetiske gyldige svar:",len(real_rows))
print("Median trust:",real_rows.trust.median())
print("Samlet antal tilføjelser:",real_rows.added.sum())
display(real_rows)

## 26. De største metodiske svagheder – svar til censor

**(A) Lille, snævert teknisk testset.** 30 held-out data scientist-opslag til main metode-test og kun 4 CV'er i den tilgængelige CV-resultatfil. Fordelen er målbar, reproducerbar evaluering; ulempen er begrænset generalisering.

**(B) Gold labels er menneskelige.** 0,86 positive-agreement F1 viser, at mennesker ikke altid er enige. Union er et dokumenteret designvalg, men ikke fejlfri sandhed.

**(C) Streng versus lempelig evaluering.** Open-vocabulary-fund straffes hårdt under strict scoring, men kan tælle som rigtige under lenient scoring. Begge tal bør vises og tolkes.

**(D) O*NET-katalogets dækningsgrad.** 47 ud af 204 labellede tools (23 %) var uden for vokabularet på testopslagene. Selv en perfekt lukket keyword-matcher vil derfor ikke kunne få fuld recall på *alle* værktøjer.

**(E) Kompromis mellem kvalitet og runtime.** LLM/Hybrid kan genfinde flere værktøjer, men API-kald er langsommere og giver data-/privatlivshensyn. Appens human-in-the-loop er en praktisk designløsning, ikke det samme som at indsætte Hybrid i produktion.

**(F) Datakildens repræsentativitet.** LinkedIn-snapshots fra 2024, mest USA, er ikke et aktuelt dansk arbejdsmarked; Goal 2 viser robusthed mellem udtræk, ikke repræsentativitet eller tidslige trends.

**(G) Brugerundersøgelsen.** Pilotfeedback førte til redesign, men man kan ikke udlede bred brugeraccept fra to pilotdeltagere. Komplet peer-trial-rapportering kræver faktiske anonymiserede svar.

## 27. Andre metoder: Hvad kunne I have valgt?

| I valgte | Alternativ | Fordel | Ulempe |
|---|---|---|---|
| Ét fast holdout (10 dev / 30 test) | Cross-validation eller flere splits | Mere stabilt performance-estimat | Tungere at gennemføre med manuelt gold og API-kald |
| To uafhængige menneskelige labels + union | Konsensus/adjudikation | Mere eksplicit løsning af uenigheder | Mere manuelt arbejde og subjektive beslutninger |
| Precision/recall/F1 (micro) | Macro-F1 pr. posting | Hvert opslag vægter ens | Kan blive ustabilt på opslag uden tools |
| Strict + lenient | Manuelt evalueret semantisk tool-match | Bedre vurdering af reelle ækvivalenser | Kræver tidskrævende adjudikation |
| Paired bootstrap | Permutationstest eller McNemar på binære fejl | Andre statistiske perspektiver | Mere kompliceret ved mange tools pr. posting |
| Keyword i app + user correction | LLM-/Hybrid i app | Bedre open-vocabulary recall | Langsommere og flere persondata/API-hensyn |
| Think-aloud og Likert | A/B-test mod alternativ app | Direkte sammenligning af design | Kræver flere deltagere |

## 28. Eksamensspørgsmål med korte svar

**1. Hvorfor har I et dev- og et testset?**  For at optimere/udvikle på dev og rapportere på et uafhængigt testset.

**2. Hvorfor 40 postings?**  Det er det fast udtrukne, manuelt labellede design (10/30), men et lille og domæneafgrænset sample.

**3. Hvad er et gold label?**  Et menneskeligt annoteret referenceværktøj i opslaget, normaliseret til den fælles repræsentation når muligt.

**4. Hvorfor to labellere?**  For at undersøge om reference-annoteringen selv er konsistent.

**5. Hvad betyder F1 = 0,86 mellem labellere?**  Der var betydelig positiv enighed, men ikke fuldstændig overensstemmelse.

**6. Hvad er forskellen på recall vocab og recall all?**  Den første ignorerer gold uden for O*NET, den anden tæller dem med.

**7. Hvorfor er strict og lenient forskellige?**  Lenient kan give credit til ens OOV-teknologier med forskellige navnformer.

**8. Hvad var keyword-matchernes største forbedring?**  Acronym-reglen øgede vocab recall med 0,39 kontra exact.

**9. Hvorfor var Hybrid nyttig?**  Under lenient scoring fandt den væsentligt flere af alle gold-tools.

**10. Hvad viser HybridVerify?**  Kontextverifikation alene forbedrede stort set ikke recall; tilføjelser drev forbedringen.

**11. Er Hybrid bevist bedst i alle situationer?**  Nej, resultaterne er fra 30 data-scientist-opslag og afhænger af evalueringsdefinitionen.

**12. Hvorfor bootstrap over items?**  For at bevare afhængighed mellem værktøjsbeslutninger inden for det samme opslag.

**13. Hvad fortæller en paired bootstrap?**  Forskellens variation mellem metoder på de samme resamplede opslag.

**14. Hvad blev frosset i LLM-evalueringen?**  Gemini-model, few-shot-eksempler og embedding-link-threshold; valgt på dev før test.

**15. Hvorfor ikke bruge den bedste LLM i webappen?**  Kode, svartid, ekstern databehandling og user correction taler for keyword som hurtig prototype.

**16. Hvad ved I om performance på CV'er?**  Den tilgængelige resultatfil evaluerer keyword-varianter på fire CV'er; LLM-CV-resultater er ikke dokumenteret dér.

**17. Hvordan testede I Goal 3?**  Pilot/think-aloud, opgaver, tool-rettelser, Likert-spørgsmål og feedback.

**18. Hvad er jeres vigtigste begrænsning?**  Små rolle-/dataspecifikke evaluation sets, ufuldstændigt O*NET-vokabular og scraperbaserede 2024-data.

## 29. Mundtlig forklaring på ca. 2 minutter

> *»Vi evaluerede projektet på tre forskellige niveauer. Først testede vi vores skill extraction med et på forhånd fastlagt udvalg på 40 data-scientist-opslag, hvor 10 var til udvikling og 30 til endelig test. To personer markerede alle navngivne teknologier uafhængigt. Vi målte en positiv agreement-F1 på 0,86, og derefter samlede vi labels til et gold-set.*
>
> *Vi målte precision, recall og F1 på de frosne matchere. Det er vigtigt, at vi skelner mellem recall på værktøjer inden for O*NET-ordforrådet og recall på alle annoterede teknologier. Den fulde keyword-matcher havde på testopslagene precision 0,97 og vocab recall 0,94, men all recall 0,73, fordi 23 % af de annoterede teknologier var uden for ordforrådet.*
>
> *I vores ablation viste acronym-normalisering sig særligt vigtig: vokabular-recall steg fra 0,56 i exact til 0,95 i acronym. Vi sammenlignede også keyword med Gemini-baseret LLMOnly, Hybrid og HybridVerify. Under lenient scoring opnåede Hybrid all recall 0,95 mod 0,73 for keyword. HybridVerify viste, at forbedringen hovedsageligt kom fra ekstra teknologier, som LLM'en fandt, snarere end at afvise forkerte kandidater. Vi rapporterer også strict scoring, som giver en anden vurdering af OOV-fundene.*
>
> *Til sidst testede vi robustheden af demand-rangeringerne og appens brugbarhed. De to pilotdeltageres feedback førte til ændringer i præsentationen af fundne tools og konfidensintervaller. Vi har dermed både en kvantitativ metodeevaluering og konkret brugerfeedback, men resultaterne kan ikke uden videre generaliseres til alle CV'er eller det danske jobmarked.*«

## 30. Oversigt over faktiske resultater og hvor du finder dem

| Påstand i din eksamensforklaring | Original kilde |
|---|---|
| 40 postings / 10 dev / 30 test | `evaluation/sample_postings.py`; `evaluation/GUIDELINES.md` |
| Agreement F1 = 0,86, kappa = 0,90 | `evaluation/GUIDELINES.md` |
| 204 labels, 47 OOV; strict keyword 0,97/0,94/0,73 | `evaluation/results/goal1_test_strict.md` |
| Acronym-effekt og ablation | `evaluation/results/goal1_test_ablation.md` |
| Lenient Hybrid all recall 0,95; CI for Δrecall | `evaluation/results/goal1_test_lenient.md` |
| Frozen LLM-konfiguration | `evaluation/llm_frozen_settings.txt` |
| Fire evaluerede CV'er og keyword metrics | `evaluation/results/goal1_cv_keyword.md` |
| Goal 2-korrelation og seniority-resultater | `analysis/results/goal2_output.txt` |
| Brugeropgaver, skalaer og consent | `evaluation/user_study.md` |
| Pilotændringer og status | `README.md` |
| Whitelist og logning af studiesvar | `skillradar/study_logging.py` |

**Links:**
- [Projektets repository](https://github.com/Lucialululu/SkillRadar)
- [Evaluation results](https://github.com/Lucialululu/SkillRadar/tree/main/evaluation/results)
- [GUIDELINES](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/GUIDELINES.md)
- [Evaluate.py](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/evaluate.py)
- [Gold merge](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/merge_gold.py)
- [LLM matcher](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/llm_matcher.py)
- [Brugertestprotokol](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/user_study.md)

**Husk:** Notebookens syntetiske eksempler er forklaret som illustrationer. De dokumenterede tal er gengivet særskilt og har specifikke kildefiler.